# 02 — Apache Airflow Fundamentals and DAG Design

## Definition
Apache Airflow is workflow orchestration software that schedules and monitors DAG-based pipelines.

## Theory
- DAG: Directed Acyclic Graph of tasks.
- Operator: executable task template (`PythonOperator`, `BashOperator`, Sensors).
- Scheduler: creates DAG runs by calendar rules.
- Executor: controls runtime strategy (`SequentialExecutor`, `LocalExecutor`, Celery, Kubernetes).

## Motivation
Cron can run commands, but cannot express lineage, retries, task-level observability, and cross-pipeline dependencies at scale.

## Tradeoff Comparison
- Cron: simple, minimal observability.
- Airflow: strong orchestration + ecosystem, heavier setup.
- Prefect: developer-friendly Python flow model.
- Dagster: asset-centric orchestration.
- Kubeflow: Kubernetes-native ML platform, higher ops overhead.

## Common Mistakes
- huge tasks that hide failure root cause
- passing large datasets via XCom
- missing retries and idempotency

In [ ]:
from pathlib import Path
from airflow.decorators import dag, task
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from airflow.sensors.filesystem import FileSensor

print('Airflow classes imported successfully')
print('DAG files:')
for p in sorted(Path('dags').glob('*.py')):
    print('-', p.name)

In [ ]:
from datetime import datetime

@dag(dag_id='tutorial_airflow_graph', start_date=datetime(2026,1,1), schedule=None, catchup=False)
def tutorial_airflow_graph():
    @task
    def extract():
        return {'rows': 200}

    @task
    def transform(meta: dict):
        return {'rows_after': meta['rows'] + 100}

    @task
    def load(meta: dict):
        print('Loaded rows:', meta['rows_after'])

    load(transform(extract()))

dag_obj = tutorial_airflow_graph()
print('Task IDs:', [t.task_id for t in dag_obj.tasks])

## Scheduling Examples
- Hourly: `0 * * * *`
- Daily: `0 6 * * *`
- Weekly retraining: `0 7 * * 1`

## Production Scheduling Best Practices
- disable catchup for live-only pipelines when backfill not needed
- use explicit SLA/runtime thresholds
- isolate training windows from peak traffic hours